In [1]:
import re, os, sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
load_dotenv()

True

In [3]:
DATASET_PATH = Path(os.getenv("PROJECT_ROOT_DIR")) / "dataset" / os.getenv("BASE_DATASET_NAME")
ROOT_FOR_FILES = Path(os.getenv("ROOT_FOR_FILES"))

In [4]:
final_dataset_df = pd.read_csv(DATASET_PATH)

In [5]:
final_dataset_df.shape

(1637, 9)

### Template for file path ###

    - decompiler->pylingual:
        - dataset->pylingual: {root_for_files}/pylingual_download/decompiler_workspace/{hash}/decompiler_output/{file}
        - dataset->pypi:  {root_for_files}/pypi_downloaded/{hash}/decompiled_output/{file}
    - decompiler->pycdc:
        - dataset->pylingual: {root_for_files}/pylingual_download/decompiler_workspace/{hash}/pycdc_decompilation_output/{file}
        - dataset->pypi: {root_for_files}/pypi_downloaded/{hash}/decompiled_output_pycdc/{file}

In [ ]:
OUTPUT_DIR = Path(os.getenv("PROJECT_ROOT_DIR")) / "dataset" / Path("subsets_of_pypi_pylingual_dataset_merged_pycdc_pylingual")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
STRAT_COLS = ["decompiler", "dataset", "bytecode_version"]

In [9]:
required_cols = [
    "file_hash", "file", "error_message", "error_description", "error",
    "bytecode_version", "decompiler", "dataset", "file_path"
]
missing = [c for c in required_cols if c not in final_dataset_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Create joint stratum
final_dataset_df["stratum"] = (
    final_dataset_df[STRAT_COLS]
    .astype(str)
    .fillna("MISSING")
    .agg("||".join, axis=1)
)

rng = np.random.default_rng(42)

# Shuffle rows within each stratum once
group_to_indices = {}
for stratum, g in final_dataset_df.groupby("stratum").groups.items():
    idxs = list(g)
    rng.shuffle(idxs)
    group_to_indices[stratum] = idxs

N = len(final_dataset_df)
base_size = N // 5
target_sizes = [base_size, base_size, base_size, base_size, N - 4 * base_size]

print("Total rows:", N)
print("Target subset sizes:", target_sizes)

subsets = []

def take_proportional_sample(group_to_indices, target_size):
    """
    Build one subset of size target_size from remaining rows.
    Try to preserve current stratum distribution as closely as possible.
    """
    remaining_counts = {k: len(v) for k, v in group_to_indices.items()}
    total_remaining = sum(remaining_counts.values())

    if target_size >= total_remaining:
        taken = []
        for k in list(group_to_indices.keys()):
            taken.extend(group_to_indices[k])
            group_to_indices[k] = []
        return taken

    # Exact proportional quotas
    exact = {
        k: (remaining_counts[k] / total_remaining) * target_size
        for k in remaining_counts
    }

    # First pass: floor
    alloc = {
        k: min(remaining_counts[k], int(np.floor(exact[k])))
        for k in remaining_counts
    }

    assigned = sum(alloc.values())
    need = target_size - assigned

    # Largest remainder method for leftover slots
    remainders = sorted(
        remaining_counts.keys(),
        key=lambda k: (exact[k] - np.floor(exact[k]), remaining_counts[k]),
        reverse=True
    )

    i = 0
    while need > 0:
        k = remainders[i % len(remainders)]
        if alloc[k] < remaining_counts[k]:
            alloc[k] += 1
            need -= 1
        i += 1

    # Actually take rows
    taken = []
    for k, n_take in alloc.items():
        if n_take > 0:
            taken.extend(group_to_indices[k][:n_take])
            group_to_indices[k] = group_to_indices[k][n_take:]

    return taken

# Build first 4 subsets as uniformly as possible
for subset_id in range(4):
    idxs = take_proportional_sample(group_to_indices, target_sizes[subset_id])
    subset_df = final_dataset_df.loc[idxs].copy().reset_index(drop=True)
    subset_df["subset_id"] = subset_id + 1
    subsets.append(subset_df)

# Last subset gets everything left
remaining_idxs = []
for k in group_to_indices:
    remaining_idxs.extend(group_to_indices[k])

subset5 = final_dataset_df.loc[remaining_idxs].copy().reset_index(drop=True)
subset5["subset_id"] = 5
subsets.append(subset5)

# Save subsets
for i, subset_df in enumerate(subsets, start=1):
    out_path = OUTPUT_DIR / f"subset_{i}.csv"
    subset_df.drop(columns=["stratum"], errors="ignore").to_csv(out_path, index=False)
    print(f"Saved {out_path} with {len(subset_df)} rows")

# Combined summary
all_subsets = pd.concat(subsets, ignore_index=True)

size_summary = (
    all_subsets.groupby("subset_id")
    .size()
    .rename("count")
    .reset_index()
)
size_summary["percent"] = size_summary["count"] / len(final_dataset_df)
size_summary.to_csv(OUTPUT_DIR / "subset_size_summary.csv", index=False)

# Joint distribution summary
joint_summary = (
    all_subsets.groupby(["subset_id"] + STRAT_COLS)
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["subset_id"] + STRAT_COLS)
)
joint_summary.to_csv(OUTPUT_DIR / "subset_joint_distribution.csv", index=False)

# Normalized summaries for quick inspection
for col in STRAT_COLS:
    summary = pd.crosstab(all_subsets["subset_id"], all_subsets[col], normalize="index")
    summary.to_csv(OUTPUT_DIR / f"distribution_{col}_by_subset.csv")
    print(f"\nDistribution of {col} by subset:")
    print(summary)

print("\nSubset sizes:")
print(size_summary)

Total rows: 1637
Target subset sizes: [327, 327, 327, 327, 329]
Saved /home/diogenes/pylingual_colaboration/pylingual_download/code/dataset/subsets_of_final_dataset/subset_1.csv with 327 rows
Saved /home/diogenes/pylingual_colaboration/pylingual_download/code/dataset/subsets_of_final_dataset/subset_2.csv with 327 rows
Saved /home/diogenes/pylingual_colaboration/pylingual_download/code/dataset/subsets_of_final_dataset/subset_3.csv with 327 rows
Saved /home/diogenes/pylingual_colaboration/pylingual_download/code/dataset/subsets_of_final_dataset/subset_4.csv with 327 rows
Saved /home/diogenes/pylingual_colaboration/pylingual_download/code/dataset/subsets_of_final_dataset/subset_5.csv with 329 rows

Distribution of decompiler by subset:
decompiler     pycdc  pylingual
subset_id                      
1           0.483180   0.516820
2           0.483180   0.516820
3           0.480122   0.519878
4           0.483180   0.516820
5           0.480243   0.519757

Distribution of dataset by subse

In [13]:
CSV_PATH = DATASET_PATH   # change this
OUT_DIR =  Path(os.getenv("PROJECT_ROOT_DIR")) / "dataset" / "eda_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

In [14]:
def extract_line_number(text: str):
    if pd.isna(text):
        return np.nan
    m = re.search(r'line\s+(\d+)', str(text), flags=re.IGNORECASE)
    return int(m.group(1)) if m else np.nan

def extract_col_number(text: str):
    if pd.isna(text):
        return np.nan
    m = re.search(r'col\s+(\d+)', str(text), flags=re.IGNORECASE)
    return int(m.group(1)) if m else np.nan

def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def version_key(v):
    try:
        parts = str(v).split(".")
        return tuple(int(p) for p in parts)
    except Exception:
        return (999, 999)

def categorize_error_family(error_message: str, error_description: str, coarse_error: str):
    msg = normalize_text(error_message).lower()
    desc = normalize_text(error_description).lower()
    coarse = normalize_text(coarse_error).lower()
    full = f"{msg} || {desc} || {coarse}"

    # String / tokenization issues
    if "unterminated f-string" in full:
        return "unterminated_fstring"
    if "unterminated string literal" in full:
        return "unterminated_string"
    if "unexpected character after line continuation character" in full:
        return "line_continuation_corruption"

    # Unclosed / mismatched delimiters
    if "was never closed" in full:
        return "unclosed_delimiter"
    if "does not match opening parenthesis" in full:
        return "delimiter_mismatch"

    # Indentation / block structure
    if "unexpected indent" in full:
        return "unexpected_indent"
    if "expected 'except' or 'finally' block" in full:
        return "missing_except_finally"
    if "'return' outside function" in full:
        return "return_outside_function"
    if "indentationerror" in full:
        return "indentation_error_other"

    # Assignment target corruption
    if "cannot assign to true" in full:
        return "assign_to_true"
    if "cannot assign to literal" in full:
        return "assign_to_literal"

    # Missing punctuation / malformed headers
    if "expected ':'" in full:
        return "missing_colon"

    # Import corruption
    if re.search(r'\bfrom\s+import\b', desc):
        return "malformed_import"
    if re.search(r'\bimport\s+\w+\s*=', desc):
        return "malformed_import"

    # Lambda / comprehension / expression corruption
    if "lambda" in desc and "invalid syntax" in full:
        return "malformed_lambda_or_comprehension"

    # Generic invalid syntax subclasses
    if "invalid syntax. perhaps you forgot a comma?" in full:
        return "invalid_syntax_missing_comma_hint"
    if "invalid syntax" in full:
        return "invalid_syntax_generic"

    # Syntax / parser fallback
    if "syntaxerror" in full:
        return "syntaxerror_other"

    return "other"

def categorize_error_group(error_family: str):
    if error_family in {
        "unterminated_fstring",
        "unterminated_string",
        "line_continuation_corruption",
    }:
        return "string_or_token_corruption"

    if error_family in {
        "unclosed_delimiter",
        "delimiter_mismatch",
    }:
        return "delimiter_structure_corruption"

    if error_family in {
        "unexpected_indent",
        "missing_except_finally",
        "return_outside_function",
        "indentation_error_other",
        "missing_colon",
    }:
        return "block_or_header_corruption"

    if error_family in {
        "assign_to_true",
        "assign_to_literal",
    }:
        return "assignment_target_corruption"

    if error_family in {
        "malformed_import",
        "malformed_lambda_or_comprehension",
        "invalid_syntax_missing_comma_hint",
        "invalid_syntax_generic",
        "syntaxerror_other",
    }:
        return "generic_parser_breakdown"

    return "other"

def extract_workspace_kind(path_str: str):
    p = normalize_text(path_str).lower()
    if "pypi_downloaded" in p:
        return "pypi_downloaded"
    if "decompiler_workspace" in p:
        return "decompiler_workspace"
    return "other"

def extract_output_kind(path_str: str):
    p = normalize_text(path_str).lower()
    if "decompiled_output_pycdc" in p:
        return "decompiled_output_pycdc"
    if "pycdc_decompilation_output" in p:
        return "pycdc_decompilation_output"
    if "decompiled_output" in p:
        return "decompiled_output"
    if "decompiler_output" in p:
        return "decompiler_output"
    return "other"

In [15]:
df = pd.read_csv(CSV_PATH)

# Basic cleanup
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.strip()

In [16]:
df["bytecode_version"] = df["bytecode_version"].astype(str)
df["line_number"] = df["error_description"].apply(extract_line_number)
df["col_number"] = df["error_description"].apply(extract_col_number)

df["error_family"] = df.apply(
    lambda r: categorize_error_family(
        r.get("error_message", ""),
        r.get("error_description", ""),
        r.get("error", "")
    ),
    axis=1
)

df["error_group"] = df["error_family"].apply(categorize_error_group)
df["workspace_kind"] = df["file_path"].apply(extract_workspace_kind)
df["output_kind"] = df["file_path"].apply(extract_output_kind)
df["file_basename"] = df["file_path"].apply(lambda x: Path(x).name if pd.notna(x) else "")
df["path_depth"] = df["file_path"].apply(lambda x: len(Path(x).parts) if pd.notna(x) else np.nan)

# More focused subtype for generic invalid syntax
invalid_mask = df["error_family"] == "invalid_syntax_generic"
desc_lower = df["error_description"].str.lower()

df["invalid_syntax_subtype"] = np.where(
    invalid_mask & desc_lower.str.contains(r"\blambda\b", na=False),
    "lambda_related",
    np.where(
        invalid_mask & desc_lower.str.contains(r"\bfrom\s+import\b", na=False),
        "import_related",
        np.where(
            invalid_mask & desc_lower.str.contains(r"\bexcept\b", na=False),
            "except_related",
            np.where(
                invalid_mask & desc_lower.str.contains(r"\bdef\b", na=False),
                "signature_or_def_related",
                np.where(
                    invalid_mask & desc_lower.str.contains(r"\breturn\b", na=False),
                    "return_related",
                    np.where(
                        invalid_mask & desc_lower.str.contains(r"\bimport\b", na=False),
                        "import_related",
                        np.where(invalid_mask, "other_invalid_syntax", "")
                    )
                )
            )
        )
    )
)


In [17]:
print("\n=== SHAPE ===")
print(df.shape)

print("\n=== MISSING VALUES ===")
print(df.isna().sum().sort_values(ascending=False))

print("\n=== UNIQUE COUNTS ===")
for c in ["file_hash", "file", "error_message", "error", "bytecode_version", "decompiler", "dataset"]:
    if c in df.columns:
        print(f"{c}: {df[c].nunique()}")

print("\n=== DUPLICATE CHECKS ===")
print("Duplicate full rows:", df.duplicated().sum())
print("Duplicate file_hash rows:", df.duplicated(subset=["file_hash"]).sum())

# How many file_hash values appear under multiple decompilers?
if {"file_hash", "decompiler"}.issubset(df.columns):
    multi_dec = (
        df.groupby("file_hash")["decompiler"]
        .nunique()
        .reset_index(name="n_decompilers")
    )
    print("file_hash appearing in multiple decompilers:", (multi_dec["n_decompilers"] > 1).sum())



=== SHAPE ===
(1637, 18)

=== MISSING VALUES ===
col_number                1637
file_hash                    0
file                         0
error_message                0
error                        0
error_description            0
decompiler                   0
dataset                      0
file_path                    0
bytecode_version             0
line_number                  0
error_family                 0
error_group                  0
workspace_kind               0
output_kind                  0
file_basename                0
path_depth                   0
invalid_syntax_subtype       0
dtype: int64

=== UNIQUE COUNTS ===
file_hash: 1337
file: 928
error_message: 292
error: 102
bytecode_version: 5
decompiler: 2
dataset: 2

=== DUPLICATE CHECKS ===
Duplicate full rows: 0
Duplicate file_hash rows: 300
file_hash appearing in multiple decompilers: 1


In [18]:
def save_freq_table(series, name, top_n=None):
    out = series.value_counts(dropna=False)
    if top_n is not None:
        out = out.head(top_n)
    out.to_csv(OUT_DIR / f"{name}.csv", header=["count"])
    print(f"\n=== {name.upper()} ===")
    print(out)

save_freq_table(df["error_family"], "error_family_counts")
save_freq_table(df["error_group"], "error_group_counts")
save_freq_table(df["error_message"], "error_message_counts", top_n=30)
save_freq_table(df["bytecode_version"], "bytecode_version_counts")
save_freq_table(df["decompiler"], "decompiler_counts")
save_freq_table(df["dataset"], "dataset_counts")
save_freq_table(df["output_kind"], "output_kind_counts")
save_freq_table(df["workspace_kind"], "workspace_kind_counts")

# -----------------------------
# Plots
# -----------------------------
def save_barplot_from_counts(counts, title, xlabel, ylabel, filename, rotate_x=False):
    plt.figure(figsize=(12, 6))
    counts.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if rotate_x:
        plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=200)
    plt.close()

# Top error families
save_barplot_from_counts(
    df["error_family"].value_counts().head(20),
    "Top Error Families",
    "Error family",
    "Count",
    "top_error_families.png",
    rotate_x=True
)

# Top raw error messages
save_barplot_from_counts(
    df["error_message"].value_counts().head(20),
    "Top Raw Error Messages",
    "Error message",
    "Count",
    "top_error_messages.png",
    rotate_x=True
)

# Bytecode version counts
version_counts = df["bytecode_version"].value_counts()
version_counts = version_counts.loc[sorted(version_counts.index, key=version_key)]
save_barplot_from_counts(
    version_counts,
    "Rows by Bytecode Version",
    "Bytecode version",
    "Count",
    "bytecode_version_counts.png",
    rotate_x=True
)

# Decompiled tool counts
save_barplot_from_counts(
    df["decompiler"].value_counts(),
    "Rows by Decompiler",
    "Decompiler",
    "Count",
    "decompiler_counts.png",
    rotate_x=False
)

# -----------------------------
# Normalized distributions
# -----------------------------
# error_family by decompiler
if {"decompiler", "error_family"}.issubset(df.columns):
    ct = pd.crosstab(df["decompiler"], df["error_family"])
    ct.to_csv(OUT_DIR / "crosstab_decompiler_error_family_raw.csv")

    ct_norm = pd.crosstab(df["decompiler"], df["error_family"], normalize="index")
    ct_norm.to_csv(OUT_DIR / "crosstab_decompiler_error_family_normalized.csv")

    plt.figure(figsize=(14, 7))
    ct_norm.plot(kind="bar", stacked=True, figsize=(14, 7))
    plt.title("Normalized Error Family Distribution by Decompiler")
    plt.xlabel("Decompiler")
    plt.ylabel("Proportion")
    plt.legend(title="Error family", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "normalized_error_family_by_decompiler.png", dpi=200)
    plt.close()

# error_group by decompiler
if {"decompiler", "error_group"}.issubset(df.columns):
    ctg_norm = pd.crosstab(df["decompiler"], df["error_group"], normalize="index")
    ctg_norm.to_csv(OUT_DIR / "crosstab_decompiler_error_group_normalized.csv")

    plt.figure(figsize=(10, 6))
    ctg_norm.plot(kind="bar", stacked=True, figsize=(10, 6))
    plt.title("Normalized Error Group Distribution by Decompiler")
    plt.xlabel("Decompiler")
    plt.ylabel("Proportion")
    plt.legend(title="Error group", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "normalized_error_group_by_decompiler.png", dpi=200)
    plt.close()

# Heatmap: decompiler x bytecode_version
if {"decompiler", "bytecode_version"}.issubset(df.columns):
    heat = pd.crosstab(df["decompiler"], df["bytecode_version"])
    heat = heat.reindex(columns=sorted(heat.columns, key=version_key))
    heat.to_csv(OUT_DIR / "heatmap_decompiler_bytecode_counts.csv")

    plt.figure(figsize=(10, 5))
    sns.heatmap(heat, annot=True, fmt="d", cmap="Blues")
    plt.title("Count of Error Rows: Decompiler x Bytecode Version")
    plt.xlabel("Bytecode version")
    plt.ylabel("Decompiler")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "heatmap_decompiler_bytecode_counts.png", dpi=200)
    plt.close()

# Heatmap: decompiler x error_family
if {"decompiler", "error_family"}.issubset(df.columns):
    heat2 = pd.crosstab(df["decompiler"], df["error_family"])
    heat2.to_csv(OUT_DIR / "heatmap_decompiler_error_family_counts.csv")

    plt.figure(figsize=(16, 5))
    sns.heatmap(heat2, annot=True, fmt="d", cmap="magma")
    plt.title("Count of Error Rows: Decompiler x Error Family")
    plt.xlabel("Error family")
    plt.ylabel("Decompiler")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "heatmap_decompiler_error_family_counts.png", dpi=200)
    plt.close()

# Heatmap: bytecode_version x error_family
if {"bytecode_version", "error_family"}.issubset(df.columns):
    heat3 = pd.crosstab(df["bytecode_version"], df["error_family"])
    heat3 = heat3.reindex(index=sorted(heat3.index, key=version_key))
    heat3.to_csv(OUT_DIR / "heatmap_bytecode_error_family_counts.csv")

    plt.figure(figsize=(16, 6))
    sns.heatmap(heat3, annot=True, fmt="d", cmap="viridis")
    plt.title("Count of Error Rows: Bytecode Version x Error Family")
    plt.xlabel("Error family")
    plt.ylabel("Bytecode version")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "heatmap_bytecode_error_family_counts.png", dpi=200)
    plt.close()

# Dataset x decompiler
if {"dataset", "decompiler"}.issubset(df.columns):
    ct_ds = pd.crosstab(df["dataset"], df["decompiler"])
    ct_ds.to_csv(OUT_DIR / "crosstab_dataset_decompiler.csv")

    plt.figure(figsize=(8, 5))
    ct_ds.plot(kind="bar", figsize=(8, 5))
    plt.title("Rows by Dataset and Decompiler")
    plt.xlabel("Dataset")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "dataset_decompiler_counts.png", dpi=200)
    plt.close()

# Line number histogram
if "line_number" in df.columns and df["line_number"].notna().any():
    plt.figure(figsize=(10, 6))
    plt.hist(df["line_number"].dropna(), bins=30)
    plt.title("Distribution of Failure Line Numbers")
    plt.xlabel("Line number")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "line_number_histogram.png", dpi=200)
    plt.close()

# Boxplot of line number by decompiler
if {"decompiler", "line_number"}.issubset(df.columns):
    tmp = df[df["line_number"].notna()].copy()
    if not tmp.empty:
        plt.figure(figsize=(8, 6))
        sns.boxplot(data=tmp, x="decompiler", y="line_number")
        plt.title("Failure Line Number by Decompiler")
        plt.xlabel("Decompiler")
        plt.ylabel("Line number")
        plt.tight_layout()
        plt.savefig(OUT_DIR / "line_number_by_decompiler_boxplot.png", dpi=200)
        plt.close()

# Invalid syntax subtype analysis
tmp_invalid = df[df["invalid_syntax_subtype"] != ""].copy()
if not tmp_invalid.empty:
    save_freq_table(tmp_invalid["invalid_syntax_subtype"], "invalid_syntax_subtype_counts")
    ct_invalid = pd.crosstab(tmp_invalid["decompiler"], tmp_invalid["invalid_syntax_subtype"], normalize="index")
    ct_invalid.to_csv(OUT_DIR / "invalid_syntax_subtype_by_decompiler_normalized.csv")

    plt.figure(figsize=(10, 6))
    ct_invalid.plot(kind="bar", stacked=True, figsize=(10, 6))
    plt.title("Invalid Syntax Subtypes by Decompiler")
    plt.xlabel("Decompiler")
    plt.ylabel("Proportion")
    plt.legend(title="Subtype", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "invalid_syntax_subtype_by_decompiler.png", dpi=200)
    plt.close()

# -----------------------------
# File-hash overlap across decompilers
# -----------------------------
if {"file_hash", "decompiler"}.issubset(df.columns):
    overlap = (
        df.assign(present=1)
          .pivot_table(index="file_hash", columns="decompiler", values="present", aggfunc="max", fill_value=0)
    )
    overlap.to_csv(OUT_DIR / "filehash_presence_by_decompiler.csv")

    # Summary: how many hashes are shared vs unique
    overlap["n_decompilers_present"] = overlap.sum(axis=1)
    overlap["n_decompilers_present"].value_counts().sort_index().to_csv(
        OUT_DIR / "filehash_num_decompilers_present_counts.csv",
        header=["count"]
    )

    print("\n=== FILE HASH OVERLAP ACROSS DECOMPILERS ===")
    print(overlap["n_decompilers_present"].value_counts().sort_index())

    # For same file_hash across multiple decompilers, compare error families
    fam_pivot = (
        df.pivot_table(index="file_hash", columns="decompiler", values="error_family", aggfunc="first")
    )
    fam_pivot.to_csv(OUT_DIR / "filehash_error_family_by_decompiler.csv")

# -----------------------------
# Representative examples per family
# -----------------------------
rep_cols = [
    c for c in [
        "file_hash", "file", "error_message", "error_family", "error_group",
        "invalid_syntax_subtype", "bytecode_version", "decompiler", "dataset",
        "line_number", "file_path"
    ] if c in df.columns
]

examples = (
    df.sort_values(["error_family", "line_number"], na_position="last")
      .groupby("error_family", as_index=False)
      .head(3)[rep_cols]
)
examples.to_csv(OUT_DIR / "representative_examples_per_error_family.csv", index=False)



=== ERROR_FAMILY_COUNTS ===
error_family
syntaxerror_other                    362
malformed_lambda_or_comprehension    290
invalid_syntax_generic               261
return_outside_function              184
unexpected_indent                    159
malformed_import                      99
delimiter_mismatch                    95
unclosed_delimiter                    43
unterminated_string                   40
invalid_syntax_missing_comma_hint     26
assign_to_literal                     16
unterminated_fstring                  15
missing_colon                         14
indentation_error_other               14
line_continuation_corruption          10
missing_except_finally                 6
assign_to_true                         3
Name: count, dtype: int64

=== ERROR_GROUP_COUNTS ===
error_group
generic_parser_breakdown          1038
block_or_header_corruption         377
delimiter_structure_corruption     138
string_or_token_corruption          65
assignment_target_corruption        19


/tmp/ipykernel_31982/1442710959.py:29: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()



=== INVALID_SYNTAX_SUBTYPE_COUNTS ===
invalid_syntax_subtype
other_invalid_syntax        170
except_related               45
signature_or_def_related     27
return_related               15
import_related                4
Name: count, dtype: int64

=== FILE HASH OVERLAP ACROSS DECOMPILERS ===
n_decompilers_present
1    1336
2       1
Name: count, dtype: int64


<Figure size 1400x700 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [19]:
# -----------------------------
# Save enriched dataframe
# -----------------------------
df.to_csv(OUT_DIR / "eda_enriched_dataset.csv", index=False)

print(f"\nDone. Outputs saved to: {OUT_DIR.resolve()}")


Done. Outputs saved to: /home/diogenes/pylingual_colaboration/pylingual_download/code/dataset/eda_outputs


In [20]:
# def build_file_path(row, root_for_files: Path):
#     """
#     Build the file path based on dataset and decompiler values.
#     """

#     file_hash = row["file_hash"]
#     file_name = row["file"]
#     decompiler = row["decompiler"]
#     dataset = row["dataset"]

#     if decompiler == "pylingual":
#         if dataset == "pylingual":
#             return root_for_files / "pylingual_download" / "decompiler_workspace" / file_hash / "decompiler_output" / file_name
#         elif dataset == "pypi":
#             return root_for_files / "pypi_downloaded" / file_hash / "decompiled_output" / file_name

#     elif decompiler == "pycdc":
#         if dataset == "pylingual":
#             return root_for_files / "pylingual_download" / "decompiler_workspace" / file_hash / "pycdc_decompilation_output" / file_name
#         elif dataset == "pypi":
#             return root_for_files / "pypi_downloaded" / file_hash / "decompiled_output_pycdc" / file_name

#     return None


# final_dataset_df["file_path"] = final_dataset_df.apply(
#     lambda row: build_file_path(row, ROOT_FOR_FILES),
#     axis=1
# )

# final_dataset_df.dropna(subset=["file_path"], inplace=True)

# def extract_filename_from_error_description(error_description: str):
#     if not isinstance(error_description, str):
#         return None

#     # Format 1: File "/path/to/file.py", line N
#     m = re.search(r'File\s+"+([^"]+)"+', error_description)
#     if m:
#         return Path(m.group(1)).name

#     # Format 2: Sorry: ... (filename.py, line N)
#     m = re.search(r'\(([^(),]+\.py),\s*line\s*\d+\)', error_description)
#     if m:
#         return Path(m.group(1)).name

#     return None


# def replace_file_for_invalid_rows(df):
#     """
#     For rows whose file_path is invalid, extract filename from error_description
#     and replace the file column with that filename.
#     """
#     df = df.copy()

#     invalid_mask = ~df["file_path"].map(lambda p: Path(p).exists())

#     df.loc[invalid_mask, "file"] = df.loc[invalid_mask, "error_description"].apply(
#         extract_filename_from_error_description
#     )

#     return df

# final_dataset_df = replace_file_for_invalid_rows(final_dataset_df)

# invalid_rows = final_dataset_df[
#     ~final_dataset_df["file_path"].map(lambda p: Path(p).exists())
# ]
# invalid_rows = final_dataset_df[
#     ~final_dataset_df["file_path"].map(lambda p: Path(p).exists())
# ]

# def normalize_bytecode_version(value):
#     try:
#         return f"{float(value):.2f}"
#     except Exception:
#         return str(value).strip()


# def move_pylingual_pypi_312_files_safely(df: pd.DataFrame, root_for_files: Path, dry_run: bool = True):
#     required_cols = {"file_hash", "file", "decompiler", "dataset", "bytecode_version"}
#     missing = required_cols - set(df.columns)
#     if missing:
#         raise ValueError(f"Missing required columns: {sorted(missing)}")

#     work_df = df.copy()
#     work_df["bytecode_version_norm"] = work_df["bytecode_version"].apply(normalize_bytecode_version)

#     mask = (
#         (work_df["decompiler"].astype(str).str.strip() == "pylingual") &
#         (work_df["dataset"].astype(str).str.strip() == "pypi") &
#         (work_df["bytecode_version_norm"] == "3.12")
#     )

#     candidates = work_df.loc[mask, ["file_hash", "file", "decompiler", "dataset", "bytecode_version"]].copy()

#     actions = []

#     for _, row in candidates.iterrows():
#         file_hash = str(row["file_hash"]).strip()
#         file_name = Path(str(row["file"]).strip()).name

#         if not file_hash or not file_name:
#             actions.append({
#                 "file_hash": file_hash,
#                 "file": file_name,
#                 "status": "skip_missing_hash_or_file",
#                 "source_path": None,
#                 "destination_path": None,
#             })
#             continue

#         hash_dir = root_for_files / "pypi_downloaded" / file_hash
#         source_path = hash_dir / file_name
#         destination_dir = hash_dir / "decompiled_output"
#         destination_path = destination_dir / file_name

#         if destination_path.exists():
#             actions.append({
#                 "file_hash": file_hash,
#                 "file": file_name,
#                 "status": "skip_destination_already_exists",
#                 "source_path": str(source_path),
#                 "destination_path": str(destination_path),
#             })
#             continue

#         if not source_path.exists():
#             actions.append({
#                 "file_hash": file_hash,
#                 "file": file_name,
#                 "status": "skip_source_missing",
#                 "source_path": str(source_path),
#                 "destination_path": str(destination_path),
#             })
#             continue

#         if not source_path.is_file():
#             actions.append({
#                 "file_hash": file_hash,
#                 "file": file_name,
#                 "status": "skip_source_not_file",
#                 "source_path": str(source_path),
#                 "destination_path": str(destination_path),
#             })
#             continue

# final_dataset_df["bytecode_version"] = final_dataset_df["bytecode_version"].astype(str)

# final_dataset_df["bytecode_version"] = final_dataset_df["bytecode_version"].replace({
#     "3.1": "3.10"
# })